In [ ]:
import pandas as pd

from results_notebook_setup import load_results, significant_models

In [ ]:
N_BOOT = 2000
results_loader = load_results(n_boot=N_BOOT)

In [ ]:
short_code_result = results_loader.short_code
short_code_result_clean = results_loader.short_code_filtered

long_code_result = results_loader.long_code
long_code_result_clean = results_loader.long_code_filtered


In [ ]:
vek = {'model_order': significant_models}

In [ ]:
def match_index(s, df):
    if not isinstance(df.index, pd.MultiIndex) or df.index.nlevels < 2:
        return s

    if df.index.nlevels > 2:
        raise RuntimeError('Matching dataframe index for a multi-index of more than 2 levels not implemented')

    index_names = df.index.names
    s_matched = pd.DataFrame({
        k: s for k in df.index.get_level_values(index_names[1]).unique()
    }).stack()
    s_matched.index.names = index_names
    return s_matched


def print_comparison_summary(comparison_df):
    print(f"\nAgreement rate: {comparison_df.agreement.sum()}/{len(comparison_df)}")

    if (~comparison_df.agreement).sum():
        print(f"Disagreement cases:\n{comparison_df[~comparison_df.agreement][
            ['original_p', 'clean_p', 'original_failed', 'clean_failed']]}")

    er = comparison_df.exclusion_rate
    print(f"\nExclusion rate max / mean: {er.max():.3f} / {er.mean():.3f}\n")


def compare(orig_res, clean_res, alpha=0.05, glmm_id='1', model_order: list[str] | None = None):
    orig_df, clean_df = [getattr(res, f'glmm{glmm_id}_results') for res in (orig_res, clean_res)]
    comparison_df = pd.DataFrame({
        'original_p': orig_df.boot_p_value,
        'clean_p': clean_df.boot_p_value
    })
    comparison_df['original_significant'] = comparison_df['original_p'] < alpha
    comparison_df['clean_significant'] = comparison_df['clean_p'] < alpha
    comparison_df['agreement'] = comparison_df['original_significant'] == comparison_df['clean_significant']

    comparison_df['estimate_diff'] = clean_df['boot_median_log'] - orig_df['boot_median_log']

    n_resp, n_clean_resp = [res.mres.variants['main'].full_data.groupby('model').size() for res in (orig_res, clean_res)]
    er = 1 - n_clean_resp / n_resp
    comparison_df['exclusion_rate'] = match_index(er, comparison_df)

    if model_order is not None:
        comparison_df.sort_index(
            level='model',
            key=lambda idx: idx.map({model: i for i, model in enumerate(model_order)}),
            inplace=True
        )

    fi_orig, fi_clean = [
        (
                df.wald_fit_failed | df.wald_singular | df.wald_nonconvergent
        ) for df in (orig_df, clean_df)
    ]

    comparison_df['original_failed'] = match_index(fi_orig, comparison_df)
    comparison_df['clean_failed'] = match_index(fi_clean, comparison_df)

    print_comparison_summary(comparison_df)

    return comparison_df


In [ ]:
compare(short_code_result, short_code_result_clean, **vek)


In [ ]:
compare(long_code_result, long_code_result_clean, **vek)

In [ ]:
compare(short_code_result, short_code_result_clean, glmm_id='2', **vek)

In [ ]:
compare(long_code_result, long_code_result_clean, glmm_id='2', **vek)
